# CS-Jail-UR: Phase 2 (Exp 7 training + Exp 8 generation, GPU)

Runs headless with **Save Version → Save & Run All (Commit)**. Design: `docs/PHASE2_PLAN.md`;
driver: `scripts/phase2_kaggle.py`.

| `STAGE` | What it does | Needs | Time |
|---|---|---|---|
| `"smoke"` | DPO on 8 harmless pairs per model (fp16 on T4, LoRA modules), then the adapter in vLLM | nothing private | ~30–40 min |
| `"train_eval"` | All adapters (C and B_ext × 3 seeds, C n-curve, D6 ablation) and Exp 8 generations, Phi-3 on GPU 0 and Llama on GPU 1 in parallel | the private pair dataset | ~8–10 h; stops itself at 11 h |

**Time guard and resume.** Kaggle kills a run at 12 h and may then lose its output, so at `STOP_AFTER_H`
the notebook stops the jobs, packages everything and ends normally (`TIME GUARD` in the log). To finish,
run it again with that version's output added as input (Add Input → Your Work → this notebook): finished
adapters, evaluations and the learning-check decision are restored and skipped.

Before saving:
1. Settings → Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets: attach **HF_TOKEN** (Llama-3.2 is gated).
3. For `"train_eval"`: Add Input → your **private** dataset made from `outputs/phase2_upload.zip`
   (verified pairs + external English pairs; harmful text). Any dataset name works: the notebook
   searches `/kaggle/input` for it.

Nothing is judged here (Kaggle cannot reach the signed-in Ollama judge). Download
`phase2_outputs.zip` (and `phase2_adapters.zip` if wanted) from the version's **Output** tab and judge
on the laptop with `exp8_posteval.py --judge-only`. Keep this notebook **private**.


In [ ]:
# Set before "Save & Run All".
STAGE = "smoke"                          # "smoke" first; "train_eval" after the pair review is applied
MODELS = ["phi3", "llama32"]             # the Phase 2 models (one GPU each in train_eval)
KEEP_ADAPTERS = True                     # also zip the trained LoRA adapters (~40 MB each)
STOP_AFTER_H = 11.0                      # time guard: stop, package, end normally (Kaggle kills at 12 h, output lost)

import glob, json, os, signal, subprocess, time
T0 = time.time()
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip results so far into /kaggle/working (overwrites): generations, manifests, smoke results."""
    keep = [p for p in ("outputs/phase2_smoke", "outputs/phase2_gate", "outputs/exp8") if os.path.exists(p)]
    keep += glob.glob("outputs/models/*/training_manifest.json")
    # capture_output: a child writing to the kernel's own stdout can block the notebook for good
    if keep:
        subprocess.run("rm -f /kaggle/working/phase2_outputs.zip && zip -qr /kaggle/working/phase2_outputs.zip "
                       + " ".join(keep) + " -x 'outputs/phase2_smoke/*/adapter/*.safetensors'", shell=True,
                       capture_output=True)
    if KEEP_ADAPTERS and os.path.isdir("outputs/models"):
        subprocess.run("rm -f /kaggle/working/phase2_adapters.zip && zip -qr /kaggle/working/phase2_adapters.zip outputs/models",
                       shell=True, capture_output=True)


def log_tail(path, lines=2, max_bytes=20000):
    """The end of a log, short and printed by Python (visible in the cell output). Progress bars
    rewrite one line with carriage returns thousands of times, so split on those too."""
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - max_bytes))
            text = f.read().decode("utf-8", "replace")
    except OSError:
        return "(no log yet)"
    parts = [p.strip()[-300:] for p in text.replace("\r", "\n").split("\n") if p.strip()]
    return "\n".join(parts[-lines:])


In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
# The training stack (TRL, PEFT, bitsandbytes) on top of the pinned vLLM stack.
sh('python -m uv pip install --python /root/py312/bin/python -e "' + '.[train,dev]' + '"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# GPU, gated-model token, frozen dataset, test suite. Any failure stops the run here.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("attach the HF_TOKEN secret: Llama-3.2 is gated")
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")


In [ ]:
# GPU smoke (PROTOCOL §9): fp16 DPO on 8 harmless pairs, then the adapter in vLLM, per model.
if STAGE == "smoke":
    for m in MODELS:
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step train")
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step serve")
        package()
    print("SMOKE PASSED for", MODELS)


In [ ]:
# Exp 7 + Exp 8 generation: one model per GPU, in parallel, resumable; packaged every 10 minutes.
if STAGE == "train_eval":
    # Finds the private bundle anywhere under /kaggle/input (zip or unpacked, any folder name/depth) and,
    # if an earlier version's output is attached as input, restores it: finished work is skipped.
    sh("python scripts/phase2_kaggle.py stage")
    sh("ls outputs/exp6 && cat outputs/exp6/review/verified_summary.json")

    class TimeUp(Exception):
        """The time guard fired: end normally so Kaggle saves this version's output."""

    def run_parallel(phase, epochs):
        """One model per GPU; package every 10 minutes; stop the run if a model fails or time is up."""
        procs = {}
        for gpu, m in enumerate(MODELS):
            env_m = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs[m] = subprocess.Popen(["python", "scripts/phase2_kaggle.py", "run", "--model", m, "--phase", phase,
                                         "--epochs", str(epochs)], env=env_m,
                                        stdout=open(f"/kaggle/working/phase2_{m}_{phase}_e{epochs}.log", "w"),
                                        stderr=subprocess.STDOUT, start_new_session=True)   # own group: the guard stops vLLM too
        while any(p.poll() is None for p in procs.values()):
            time.sleep(60 if phase == "gate" else 600)
            if time.time() - T0 > STOP_AFTER_H * 3600:
                for p in procs.values():
                    if p.poll() is None:
                        os.killpg(p.pid, signal.SIGTERM)
                time.sleep(30)
                package()
                raise TimeUp
            package()
            for m in MODELS:
                print(f"--- {m} {phase} e{epochs} ---\n" + log_tail(f"/kaggle/working/phase2_{m}_{phase}_e{epochs}.log"),
                      flush=True)
        package()
        bad = {m: p.returncode for m, p in procs.items() if p.returncode}
        if bad:
            for m in bad:   # the real error is in the log, not in this cell's traceback
                print(f"=== {m} {phase} e{epochs}: last 80 log lines ===\n"
                      + log_tail(f"/kaggle/working/phase2_{m}_{phase}_e{epochs}.log", lines=80, max_bytes=200000),
                      flush=True)
            raise RuntimeError(f"{phase} failed: {bad}; see /kaggle/working/phase2_<model>_{phase}_e{epochs}.log")

    # Learning check (configs/dpo.yaml learning_check, declared before training): seed-42 C and B_ext
    # adapters on their TRAINING logs; climb the epochs ladder until all pass (or its top). A resumed
    # run reuses the earlier version's final decision (outputs/phase2_gate/decision.json).
    ladder = json.loads(subprocess.run(["python", "-c", "import yaml, json; print(json.dumps(yaml.safe_load("
                                        "open('configs/dpo.yaml'))['learning_check']['epochs_ladder']))"],
                                       capture_output=True, text=True, check=True).stdout)
    dec_path = "outputs/phase2_gate/decision.json"
    dec = json.load(open(dec_path)) if os.path.exists(dec_path) else {}
    try:
        if dec.get("all_pass") or dec.get("epochs") == ladder[-1]:
            epochs = dec["epochs"]
            print(f"RESUMED: the learning check already decided {epochs} epochs ({dec_path})", flush=True)
        else:
            for epochs in ladder:
                run_parallel("gate", epochs)
                d = subprocess.run(["python", "scripts/phase2_kaggle.py", "decide", "--models", *MODELS,
                                    "--epochs", str(epochs)], capture_output=True, text=True)
                print(d.stdout + d.stderr, flush=True)
                if d.returncode not in (0, 10):
                    raise RuntimeError(f"decide crashed (exit {d.returncode})")
                if d.returncode == 0:
                    break
                print(f"LEARNING CHECK FAILED at {epochs} epochs" + ("; retraining at the next value"
                      if epochs != ladder[-1] else "; top of the ladder reached, continuing with it (reported)"),
                      flush=True)
        print(f"USING {epochs} EPOCHS FOR ALL PHASE 2 ADAPTERS", flush=True)
        run_parallel("rest", epochs)
        print("TRAIN_EVAL DONE")
    except TimeUp:
        print(f"TIME GUARD: stopped after {STOP_AFTER_H} h so Kaggle keeps this version's output. To finish, re-run "
              "with THIS version's output added as input (Add Input -> Your Work -> this notebook); it continues "
              "where it stopped.", flush=True)


In [ ]:
package()
sh("ls -lh /kaggle/working/*.zip")
print("DONE: download phase2_outputs.zip (and phase2_adapters.zip) from this version's Output tab.")
